In [1]:
import pandas as pd

# Load Dataset
df = pd.read_csv("News Classification Labeling - Sheet1.csv")

# First 5 rows
print(df.head())

print("\n---------------------------")

# Shape
print("Dataset Shape:", df.shape)

print("\n---------------------------")

# Columns
print("Columns:")
print(df.columns)

print("\n---------------------------")

# Data Types
print(df.dtypes)

print("\n---------------------------")

# Missing Values
print(df.isnull().sum())

print("\n---------------------------")

# Category Distribution
print(df["Category"].value_counts())

                                                 URL  \
0  https://asiatimes.com/2025/10/vietnam-airlines...   
1  https://asiatimes.com/2025/10/doj-seizes-15-bi...   
2  https://www.fool.com/investing/2025/10/15/too-...   
3  https://www.fool.com/investing/2025/10/15/inte...   
4  https://www.fool.com/investing/2025/10/15/3-ro...   

                                               Title    Category Group Leader  
0  Vietnam Airlines data leak exposes a crisis of...  Technology          NaN  
1  DOJ seizes $15 billion bitcoin in SE Asia cryp...     Markets          NaN  
2  Think It's Too Late to Buy IonQ? Here's the 1 ...     Markets          NaN  
3  Intel's Crucial Panther Lake Chips Start Produ...  Technology          NaN  
4                3 Robotics Stocks to Buy in October     Markets          NaN  

---------------------------
Dataset Shape: (5000, 4)

---------------------------
Columns:
Index(['URL', 'Title', 'Category', 'Group Leader'], dtype='object')

----------------------

In [2]:
import pandas as pd
import numpy as np

# Remove unnecessary column
df = df.drop(columns=["Group Leader"])

# Remove rows with missing target
df = df.dropna(subset=["Category"])

# Remove duplicate headlines
df = df.drop_duplicates(subset=["Title"])

# Reset index
df.reset_index(drop=True, inplace=True)

####################################################
# Basic Feature Engineering
####################################################

# Character Count
df["char_count"] = df["Title"].str.len()

# Word Count
df["word_count"] = df["Title"].str.split().apply(len)

# Average Word Length
df["avg_word_length"] = df["Title"].apply(
    lambda x: np.mean([len(i) for i in x.split()])
)

# Uppercase Word Count
df["uppercase_words"] = df["Title"].apply(
    lambda x: sum(word.isupper() for word in x.split())
)

print(df.head())

print("\nShape :", df.shape)

                                                 URL  \
0  https://asiatimes.com/2025/10/vietnam-airlines...   
1  https://asiatimes.com/2025/10/doj-seizes-15-bi...   
2  https://www.fool.com/investing/2025/10/15/too-...   
3  https://www.fool.com/investing/2025/10/15/inte...   
4  https://www.fool.com/investing/2025/10/15/3-ro...   

                                               Title    Category  char_count  \
0  Vietnam Airlines data leak exposes a crisis of...  Technology          59   
1  DOJ seizes $15 billion bitcoin in SE Asia cryp...     Markets          58   
2  Think It's Too Late to Buy IonQ? Here's the 1 ...     Markets          75   
3  Intel's Crucial Panther Lake Chips Start Produ...  Technology          51   
4                3 Robotics Stocks to Buy in October     Markets          35   

   word_count  avg_word_length  uppercase_words  
0           9         5.666667                0  
1          11         4.363636                2  
2          15         4.066667  

In [3]:
import spacy

nlp = spacy.load("en_core_web_sm")

def entity_count(text):
    doc = nlp(text)
    return len(doc.ents)

df["entity_count"] = df["Title"].apply(entity_count)

print(df[["Title","entity_count"]].head())

                                               Title  entity_count
0  Vietnam Airlines data leak exposes a crisis of...             1
1  DOJ seizes $15 billion bitcoin in SE Asia cryp...             2
2  Think It's Too Late to Buy IonQ? Here's the 1 ...             1
3  Intel's Crucial Panther Lake Chips Start Produ...             2
4                3 Robotics Stocks to Buy in October             3


In [4]:
print(df.columns)

Index(['URL', 'Title', 'Category', 'char_count', 'word_count',
       'avg_word_length', 'uppercase_words', 'entity_count'],
      dtype='object')


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1,3),
    max_features=50000,
    sublinear_tf=True,
    min_df=2
)

X = tfidf.fit_transform(df["Title"])

print("TF-IDF Shape:", X.shape)

TF-IDF Shape: (4577, 14367)


In [6]:
y = df["Category"]

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words="english",
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_text = tfidf.fit_transform(df["Title"])

print("TF-IDF Shape:", X_text.shape)

TF-IDF Shape: (4577, 5000)


In [16]:


from scipy.sparse import hstack, csr_matrix

# Numerical Features
numerical_features = df[
    [
        "word_count",
        "char_count",
        "avg_word_length",
        "uppercase_words",
        "entity_count"
    ]
]

# Convert directly to Sparse Matrix (NO SCALING)
X_num = csr_matrix(numerical_features.values)

# Combine TF-IDF + Numerical Features
X = hstack([X_text, X_num])

print("Final Shape:", X.shape)

Final Shape: (4577, 5005)


In [17]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(df["Category"])

print("Classes:\n")

for i, cls in enumerate(label_encoder.classes_):
    print(f"{i} --> {cls}")

Classes:

0 --> Business
1 --> Energy
2 --> Health
3 --> Markets
4 --> Politics
5 --> Technology


In [18]:
from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [19]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

# Models

svm_model = LinearSVC(
    class_weight="balanced",
    random_state=42
)

lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

nb_model = MultinomialNB()

# Train

svm_model.fit(X_train, y_train)
lr_model.fit(X_train, y_train)
nb_model.fit(X_train, y_train)

print("All Models Trained Successfully.")

C:\Users\pk\AppData\Roaming\Python\Python310\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


All Models Trained Successfully.


C:\Users\pk\AppData\Roaming\Python\Python310\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [20]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

models = {
    "Linear SVM": svm_model,
    "Logistic Regression": lr_model,
    "Multinomial NB": nb_model
}

results = []

for name, model in models.items():

    pred = model.predict(X_val)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, pred),
        "Precision": precision_score(y_val, pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_val, pred, average="weighted", zero_division=0),
        "F1 Score": f1_score(y_val, pred, average="weighted", zero_division=0)
    })

results_df = pd.DataFrame(results)

print(results_df)
results_df = pd.DataFrame(results)

print(results_df.sort_values(by="F1 Score", ascending=False))

                 Model  Accuracy  Precision    Recall  F1 Score
0           Linear SVM  0.280932   0.626695  0.280932  0.340736
1  Logistic Regression  0.612809   0.618125  0.612809  0.613899
2       Multinomial NB  0.503639   0.586558  0.503639  0.410539
                 Model  Accuracy  Precision    Recall  F1 Score
1  Logistic Regression  0.612809   0.618125  0.612809  0.613899
2       Multinomial NB  0.503639   0.586558  0.503639  0.410539
0           Linear SVM  0.280932   0.626695  0.280932  0.340736


In [21]:
print(X.dtype)
print(X.shape)
print(df["Category"].value_counts())

float64
(4577, 5005)
Category
Business      1724
Markets       1388
Politics       816
Technology     398
Health         134
Energy         117
Name: count, dtype: int64
